In [30]:
import numpy as np
import matplotlib.pyplot as plt
import time
import cv2

## Hồng Gấm – Demo 1: Ảnh là bảng số

In [ ]:
img = cv2.imread("input.jpg")

if img is None:
    raise FileNotFoundError("Không tìm thấy input.jpg")

img = cv2.resize(img, (200, 200))
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(5, 5))
plt.imshow(img_rgb)
plt.xlabel("Original Image", fontsize=12, fontweight="bold")
plt.xticks([])
plt.yticks([])
plt.show()

In [ ]:
B, G, R = cv2.split(img)

channels = [B, G, R]
labels = ["Blue Channel", "Green Channel", "Red Channel"]

plt.figure(figsize=(12, 4))

for i in range(3):
    plt.subplot(1, 3, i + 1)
    plt.imshow(channels[i], cmap="gray", vmin=0, vmax=255)
    plt.xlabel(labels[i], fontsize=12, fontweight="bold")
    plt.xticks([])
    plt.yticks([])

plt.tight_layout()
plt.show()

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

row = gray.shape[0] // 2

plt.figure(figsize=(10, 4))
plt.plot(gray[row, :])

plt.xlabel("Pixel Position", fontsize=12)
plt.ylabel("Brightness", fontsize=12)
plt.title("Brightness of the Middle Row")

plt.grid(True)
plt.show()

## Demo 3a: convolve2d, lọc hộp, Gaussian

In [34]:
def convolve2d(image, kernel, padding="constant"):
    image = np.asarray(image, dtype=np.float64)
    kernel = np.asarray(kernel, dtype=np.float64)

    if image.ndim != 2 or kernel.ndim != 2:
        raise ValueError("Image và kernel phải là mảng 2 chiều")

    kh, kw = kernel.shape

    if kh % 2 == 0 or kw % 2 == 0:
        raise ValueError("Kernel phải có kích thước lẻ")

    if padding not in ("constant", "reflect", "edge"):
        raise ValueError("Padding không hợp lệ")

    ph, pw = kh // 2, kw // 2

    padded = np.pad(
        image,
        ((ph, ph), (pw, pw)),
        mode=padding
    )

    flipped = np.flip(kernel, axis=(0, 1))

    h, w = image.shape
    output = np.zeros((h, w), dtype=np.float64)

    for i in range(h):
        for j in range(w):
            region = padded[i:i+kh, j:j+kw]
            output[i, j] = np.sum(region * flipped)

    return output

In [ ]:
pad = 20

constant = np.pad(
    img, ((pad, pad), (pad, pad), (0, 0)),
    mode="constant"
)

reflect = np.pad(
    img, ((pad, pad), (pad, pad), (0, 0)),
    mode="reflect"
)

edge = np.pad(
    img, ((pad, pad), (pad, pad), (0, 0)),
    mode="edge"
)

images = [constant, reflect, edge]
labels = ["Constant Padding", "Reflect Padding", "Edge Padding"]

plt.figure(figsize=(12, 4))

for i, image in enumerate(images):
    plt.subplot(1, 3, i + 1)
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.xlabel(labels[i], fontsize=12, fontweight="bold")
    plt.xticks([])
    plt.yticks([])

plt.tight_layout()
plt.show()

In [ ]:
box_kernel = np.ones((15, 15)) / 225

g = cv2.getGaussianKernel(15, 4)
gaussian_kernel = g @ g.T

box_blur = np.stack(
    [convolve2d(img[:, :, c], box_kernel, "reflect") for c in range(3)],
    axis=2
)

gaussian_blur = np.stack(
    [convolve2d(img[:, :, c], gaussian_kernel, "reflect") for c in range(3)],
    axis=2
)

box_blur = np.clip(box_blur, 0, 255).astype(np.uint8)
gaussian_blur = np.clip(gaussian_blur, 0, 255).astype(np.uint8)

images = [img, box_blur, gaussian_blur]
labels = ["Original Image", "Box Filter", "Gaussian Filter"]

plt.figure(figsize=(15, 5))

for i, image in enumerate(images):
    plt.subplot(1, 3, i + 1)
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.xlabel(labels[i], fontsize=12, fontweight="bold")
    plt.xticks([])
    plt.yticks([])

plt.tight_layout()
plt.show()

## KT3 – Thí nghiệm Hough